In [ ]:
import numpy as np
import subprocess
import os
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from matplotlib.pyplot import figure, show, subplots
from matplotlib import pyplot as plt

#import yaml
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True  # Keep quotes if they exist
yaml.indent(mapping=2, sequence=4, offset=2)  # Preserve indentation
yaml.representer.add_representer(
    float, lambda dumper, data: dumper.represent_scalar("tag:yaml.org,2002:float", str(data))
)

with open("../config/masterconfig.yaml", "r") as file:
    config = yaml.load(file) or {}
dataserver_path = config['dataserver_path']
observingdistances = config['observingdistances']

In [ ]:
# get AGNI data
file="../output/AGNI_atmospherecomposition_normalized"
data = np.genfromtxt(file,skip_header=1,delimiter=",")
datalength = len(data)
titles = np.genfromtxt(file,delimiter=",",dtype='str',skip_footer=datalength)

# remove S8 because ARCiS does not support it; then renormalize
data = data[:,:-1]
new_data = []
for i in data:
    coords = [i[0],i[1]]
    normal_data = i[2:]/sum(i[2:])
    for j in normal_data:
        coords.append(j)
    new_data.append(coords)
data = new_data
titles = titles[:-1]

# make a dictionary
atmosphere_dict = {(i[0],i[1]): i[2:] for i in data}

In [ ]:
# reformat data so ARCiS can read it
def format_float(x):
    base, exp = f"{x:.2e}".split("e")
    exp = int(exp)
    return f"{base}d{exp}"
formatted = [format_float(v) for v in atmosphere_dict[(-4.0,0.1)]]

# ARCiS runs are started at a remote computing cluster and managed in tmux panes
def ARCiS_file_generator(fo2, CH, AGNI_atmos, distance):
    """Generate planetary emission spectra using ARCiS"""

    formatted = [format_float(i) for i in AGNI_atmos]
    config_ARCiS_loc = "config/forward_model_ARCiS.in"
    atmosphere_key = 'fm_ARCiS' + str(fo2) + '_' + str(CH) + '_' + str(distance)
    save_location = dataserver_path + f"/ARCiS_forwardmodels/{atmosphere_key}"
    command = (
        f"ARCiS {config_ARCiS_loc} -o {save_location} -s distance={distance}d0 "
        f"-s N2={formatted[0]} "
        f"-s CO={formatted[1]} "
        f"-s H2={formatted[2]} "
        f"-s CH4={formatted[3]} "
        f"-s H2S={formatted[4]} "
        f"-s S2={formatted[5]} "
        f"-s SO2={formatted[6]} "
        f"-s H2O={formatted[7]} "
        f"-s CO2={formatted[8]} "
        f"-s NH3={formatted[9]} "
        f"-s OCS={formatted[10]} "
    )
    print(repr(command))
      
    atmosphere_key_us = atmosphere_key.replace(".","_")
    prompt = (
        "ssh -n -T norma2;"
        "cd .. ;"
        f"tmux new-session -d -s '{atmosphere_key}'; "
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load ARCiS' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module unload cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'module load cfitsio' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 'export OMP_NUM_THREADS=8' C-m ;"
        f"tmux send-keys -t {atmosphere_key_us}:0 '{command}' C-m"
        )
    subprocess.run(prompt, shell=True, check=True) 

In [ ]:
for keys in list(atmosphere_dict.keys()):
    for distance in observingdistances:
        ARCiS_file_generator(keys[0],keys[1],atmosphere_dict[(keys[0],keys[1])],distance)

In [ ]:
# raster plot of ARCiS forward model spectra

data_path = dataserver_path + "/ARCiS_forwardmodels"
files = [f for f in os.listdir(data_path)]

mpl.rcParams['figure.dpi'] = 800


# setup custom order for rasterplot, since the subplots go left->right, up->down
# but we need to go down->up, left->right
custom_order = [9,5,1,10,6,2,11,7,3,12,8,4]

for select_distance in observingdistances:
        custom_count = 0
        rasterplot = figure(figsize=(8,6))
        # do plots
        for i in range(len(done)):
                if done[i].split("_")[-1] == str(select_distance):
                        frame = rasterplot.add_subplot(3,4,custom_order[custom_count])
                        bandcenter = np.genfromtxt(data_location+done[i]+"/emis",usecols=0,skip_header=1)
                        bandflux = np.genfromtxt(data_location+done[i]+"/emis",usecols=1,skip_header=1)
                        bandcenter, bandflux = spectrum_cutoff(bandcenter,bandflux)
                        frame.plot(bandcenter,bandflux,linewidth=0.5,label='known')
                        frame.set_yscale('log')
                        frame.set_xlim(4,18.5)

                        # change ylim per distance because the intensity differs
                        if select_distance == 5:
                                frame.set_ylim(5e-10,5e-06)
                        elif select_distance == 10:
                                frame.set_ylim(10e-11,1.8e-06)
                        elif select_distance == 15:
                                frame.set_ylim(5e-11,8e-7)
                        elif select_distance == 20:
                                frame.set_ylim(2e-11,5e-07)
                        
                        plt.tick_params(
                        axis='both',      # both x and y axes
                        which='both',     # major and minor ticks
                        labelsize=5.5,      # ↓ font size of tick labels
                        length=2,         # ↓ length of tick marks
                        width=0.4,         # ↓ thickness of tick marks
                        pad=1.2
                        )
                        frame.minorticks_off()
                        
                        
                        custom_count += 1
                

        # add axes to the plot (I feel like I over-engineered this)
        master_ax = rasterplot.add_axes([0.09, 0.08, 0.82, 0.82])
        master_ax.patch.set_alpha(0)
        master_ax.set_zorder(-10)

        master_ax.spines['top'].set_visible(False)
        master_ax.spines['right'].set_visible(False)
        master_ax.spines['left'].set_linewidth(1.5)
        master_ax.spines['bottom'].set_linewidth(1.5)

        y_centers = [0.83, 0.50, 0.17] 
        master_ax.set_ylim(0, 1)
        master_ax.set_yticks(y_centers)
        master_ax.set_yticklabels(["5", "1", "0.1"])

        x_centers = [0.135, 0.365, 0.635, 0.865]
        master_ax.set_xlim(0, 1)
        master_ax.set_xticks(x_centers)
        master_ax.set_xticklabels(["-4", "-1.3", "1.3", "4"])

        master_ax.tick_params(labelsize=12)


        rasterplot.supxlabel('Mantle $f$O$_2$, $\Delta$ IW')
        rasterplot.supylabel('Mantle C/H mass ratio')
        rasterplot.show()
        rasterplot.savefig('../output/figures/ARCiS_fm_raster_{}pc.png'.format(select_distance))